# Pomegranate Orchard Monitor — Full Pipeline
## Training: Dataset → YOLO → U-Net → Model Export

**Dataset:** Makinay's Pomegranate (Roboflow) — 7k images, Instance Segmentation  
**URL:** https://universe.roboflow.com/makinay-pomagranate/pomegranate-y3sbw

In [1]:
# SETUP: Install dependencies
!pip install -q ultralytics roboflow segmentation-models-pytorch albumentations grad-cam pycocotools

import os, yaml, cv2, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 59.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 80.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


In [2]:
# STEP 1: Download Dataset from Roboflow
from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
ROBOFLOW_API_KEY = user_secrets.get_secret("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("makinay-pomagranate").project("pomegranate-y3sbw")
dataset = project.version(1).download("yolov8")
print(f"Dataset downloaded to: {dataset.location}")
DATA_DIR = dataset.location

ConnectionError: Connection error trying to communicate with service.

In [ ]:
# STEP 2: Data Exploration
import glob

with open(os.path.join(DATA_DIR, 'data.yaml'), 'r') as f:
    data_cfg = yaml.safe_load(f)
print("Classes:", data_cfg.get('names', 'N/A'))
print("Train:", data_cfg.get('train'))
print("Val:", data_cfg.get('val'))

train_imgs = len(glob.glob(os.path.join(DATA_DIR, 'train/images/*'))) if data_cfg.get('train') else 0
val_imgs = len(glob.glob(os.path.join(DATA_DIR, 'valid/images/*'))) if data_cfg.get('val') else 0
print(f"Train images: {train_imgs}, Val images: {val_imgs}")

In [ ]:
# STEP 3: Train YOLOv8 Detection
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data=os.path.join(DATA_DIR, 'data.yaml'),
    epochs=80,
    patience=15,
    imgsz=640,
    batch=16,
    lr0=0.01,
    lrf=0.01,
    optimizer='AdamW',
    cos_lr=True,
    mosaic=0.5,
    project='/kaggle/working/results',
    name='yolo_detection',
    exist_ok=True,
)

metrics = model.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

os.makedirs('/kaggle/working/weights', exist_ok=True)
model.save('/kaggle/working/weights/yolov8_pomegranate.pt')

In [ ]:
# STEP 4: Download in COCO format (for segmentation masks) & extract crops
# The yolov8 format only has bboxes — COCO includes polygon masks
coco_dataset = project.version(1).download("coco")
print(f"COCO dataset at: {coco_dataset.location}")
COCO_DIR = coco_dataset.location

def extract_crops_and_masks(data_dir, output_dir, split='train'):
    img_dir = os.path.join(data_dir, split)
    ann_file = os.path.join(data_dir, split, '_annotations.coco.json')
    
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'masks'), exist_ok=True)
    
    with open(ann_file, 'r') as f:
        coco = json.load(f)
    
    img_map = {img['id']: img['file_name'] for img in coco['images']}
    cat_map = {cat['id']: cat['name'] for cat in coco['categories']}
    print(f"Categories: {cat_map}")
    
    crop_idx = 0
    for ann in coco['annotations']:
        img_id = ann['image_id']
        img_file = img_map.get(img_id)
        if img_file is None:
            continue
        img_path = os.path.join(img_dir, img_file)
        if not os.path.exists(img_path):
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Get bbox from annotation
        x, y, bw, bh = ann['bbox']
        x1 = max(0, int(x))
        y1 = max(0, int(y))
        x2 = min(w, int(x + bw))
        y2 = min(h, int(y + bh))
        if x2 <= x1 or y2 <= y1:
            continue
        
        # Build mask from polygon segmentation
        mask = np.zeros((h, w), dtype=np.uint8)
        if 'segmentation' in ann and ann['segmentation']:
            for seg in ann['segmentation']:
                if len(seg) < 6:
                    continue
                poly = np.array(seg).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask, [poly], 255)
        
        # If no polygon, fall back to bbox mask
        if mask.sum() == 0:
            cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
        
        crop = img[y1:y2, x1:x2]
        mask_crop = mask[y1:y2, x1:x2]
        
        crop_file = f"crop_{crop_idx:06d}.jpg"
        mask_file = f"crop_{crop_idx:06d}.png"
        cv2.imwrite(os.path.join(output_dir, 'images', crop_file), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(output_dir, 'masks', mask_file), mask_crop)
        crop_idx += 1
    
    print(f"Extracted {crop_idx} crops to {output_dir}")

extract_crops_and_masks(COCO_DIR, '/kaggle/working/seg_data/train', 'train')
extract_crops_and_masks(COCO_DIR, '/kaggle/working/seg_data/val', 'valid')

In [ ]:
# STEP 5: Train U-Net Segmentation
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
from tqdm import tqdm
import albumentations as A

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

class SegDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, imgsz=256):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.imgsz = imgsz
        self.ids = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png'))])
    
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        img = cv2.imread(os.path.join(self.img_dir, self.ids[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        name = os.path.splitext(self.ids[idx])[0]
        mask = cv2.imread(os.path.join(self.mask_dir, f'{name}.png'), cv2.IMREAD_GRAYSCALE)
        
        img = cv2.resize(img, (self.imgsz, self.imgsz))
        mask = cv2.resize(mask, (self.imgsz, self.imgsz), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)
        
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        
        img = img / 255.0
        img = (img - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])
        img = torch.from_numpy(img).permute(2,0,1).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()
        return img, mask

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
])

train_ds = SegDataset('/kaggle/working/seg_data/train/images', '/kaggle/working/seg_data/train/masks', train_tf)
val_ds = SegDataset('/kaggle/working/seg_data/val/images', '/kaggle/working/seg_data/val/masks')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")

model = smp.Unet('resnet34', encoder_weights='imagenet', in_channels=3, classes=1, activation='sigmoid').to(DEVICE)
criterion = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

best_dice = 0
for epoch in range(50):
    model.train()
    train_loss = 0
    for imgs, msks in tqdm(train_loader, desc=f'E{epoch+1} Train'):
        imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), msks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_dice = 0
    with torch.no_grad():
        for imgs, msks in val_loader:
            imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
            pred = model(imgs)
            pred_bin = (pred > 0.5).float()
            inter = (pred_bin * msks).sum()
            val_dice += (2*inter/(pred_bin.sum()+msks.sum()+1e-7)).item()
    val_dice /= len(val_loader)
    
    print(f"E{epoch+1:3d} | Loss: {train_loss/len(train_loader):.4f} | Dice: {val_dice:.4f}")
    scheduler.step(val_dice)
    
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), '/kaggle/working/weights/unet_defect.pth')
        print(f"  -> Saved (Dice: {best_dice:.4f})")

print(f"Done! Best Dice: {best_dice:.4f}")

In [ ]:
# STEP 6: Package trained weights for download
import shutil
shutil.make_archive('/kaggle/working/pomegranate_models', 'zip', '/kaggle/working/weights')
print("Done! Download from /kaggle/working/pomegranate_models.zip")